<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/iLLaDA_GSM8K_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# iLLaDA on GSM8K

Evaluate random MDM, token-probability confidence, Top-K Margin, and majority-vote Best-of-5 using the pinned **GSAI-ML/iLLaDA-8B-Instruct** checkpoint. Follow the setup once, then run the decoder sections you want. This notebook checks out **adi**; the new notebook and scripts must be available on that branch.

Use a CUDA runtime with BF16 support and enough memory for unquantized 8B weights. Each evaluation loads one model in a fresh subprocess; the notebook kernel does not load the weights. Results and model caches are saved to Drive. All methods share the same GSM8K test manifest, prompts, seed assignments, and scoring as LLaDA.

Defaults: **temperature 1.0**, **256 output positions**, **256 denoising steps**, **128-token blocks**, **seed 1**, and **20 smoke questions**. The fixed-length sampler uses iLLaDA's own chat template and mask token **5**. Variable-length generation and MCTS are not enabled.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from `adi`

The private-repository checkout uses Colab Secrets (`GITHUB_TOKEN`) or a hidden prompt for a token with read access. Existing local changes or divergent commits stop the update without discarding work.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


## 2. Install iLLaDA dependencies

Use the iLLaDA dependency profile before evaluating this model. Keep Colab's CUDA-enabled PyTorch. If switching from the LLaDA notebook in the same runtime, rerun this cell first.

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements/illada-colab.txt"),
], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


## 3. Mount Drive

The iLLaDA cache is separate from the original LLaDA cache. `COPY_MODEL_TO_RUNTIME = True` stages the cached snapshot onto local runtime storage for loading.

In [ ]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "illada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "GSM8K")


## 4. Load the iLLaDA GSM8K configuration

All four evaluation commands use this model, cache, output root, and frozen dataset manifest. The manifest is shared with LLaDA and published without hard links, so it can remain on mounted Drive.

In [ ]:
import yaml

base_config = yaml.safe_load((REPO_DIR / "configs/gsm8k/illada_decoding.yaml").read_text())
base_config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH),
                            device="cuda", local_files_only=True)
base_config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
base_config["experiment_root"] = str(DRIVE_ROOT)
base_config["experiment_timezone"] = EXPERIMENT_TIMEZONE
BENCH_CONFIG = Path("/content/illada-gsm8k-decoding.yaml")
MANIFEST_PATH = DRIVE_ROOT / "manifests" / "gsm8k_test.json"
print("Model revision:", profile["revision"])
print("Mask token ID:", profile["mask_id"])
print("Shared dataset manifest:", MANIFEST_PATH)


## 5. Cache the model on Drive

The downloader pins the registered iLLaDA revision, includes the checkpoint chat template and source metadata, and reuses existing files. Staging copies the same identity into the runtime model directory.

In [ ]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


## Benchmark and dataset configuration

Choose `PROFILE = "smoke"` for 20 questions, `"initial"` for 100, or `"full"` for all 1,319 GSM8K test questions. `NUM_SAMPLES = None` uses the profile default; set a positive count to override it. Use 1 or 10 for a shorter first check if needed.

Change these controls and rerun this cell before evaluating. The prompt instructions come from the YAML configuration and are identical to the LLaDA benchmark instructions.

In [ ]:
import copy
from dlm_project.tasks.gsm8k.benchmark import resolve_settings

PROFILE = "smoke"
NUM_SAMPLES = None
TEMPERATURE = 1.0
SEED = 1
GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 128

cfg = copy.deepcopy(base_config)
cfg["data"]["num_samples"] = NUM_SAMPLES
cfg["evaluation"]["profile"] = PROFILE
cfg["generation"].update(gen_length=GEN_LENGTH, steps=STEPS,
                          block_length=BLOCK_LENGTH, temperature=TEMPERATURE)
cfg["decoding"]["seed"] = SEED
# Validate without replacing the nullable sample count with a profile-specific count.
resolved = resolve_settings("confidence", cfg, expected_family="illada")
BENCH_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(BENCH_CONFIG.read_text())
print("Selected questions:", resolved["data"]["num_samples"])


## Run and inspect helpers

Each decoder run streams its output and actual Python traceback. A successful run writes a unique receipt identifying its exact output directory. Preview cells use that receipt's directory rather than searching for a possibly unrelated previous run.

Rerun a preview cell to see another random selection of saved answers without generating again.

In [ ]:
import json
import random
import uuid

RUN_DIRS = {}
METHODS = ("confidence", "random_mdm", "top_k_margin", "best_of_5")

def run_decoding(method):
    if method not in METHODS:
        raise ValueError(f"Unknown decoding method: {method}")
    RUN_DIRS.pop(method, None)
    receipt_path = Path("/content") / f"illada-gsm8k-{method}-{uuid.uuid4().hex}.json"
    command = [
        sys.executable, "-u",
        str(REPO_DIR / "scripts" / f"eval_illada_gsm8k_{method}.py"),
        "--config", str(BENCH_CONFIG), "--profile", PROFILE,
        "--manifest", str(MANIFEST_PATH), "--run-manifest", str(receipt_path),
    ]
    if NUM_SAMPLES is not None:
        command += ["--num-samples", str(NUM_SAMPLES)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    receipt = json.loads(receipt_path.read_text())
    if (receipt["status"] != "completed" or receipt["model_family"] != MODEL_FAMILY
            or receipt["method"] != method or receipt["profile"] != PROFILE):
        raise RuntimeError("The run receipt does not match this evaluation.")
    run_dir = Path(receipt["run_dir"])
    RUN_DIRS[method] = run_dir
    print("This run:", run_dir)
    return run_dir


def show_random_answers(method, count=5):
    if type(count) is not int or count < 1:
        raise ValueError("count must be a positive integer.")
    if method not in RUN_DIRS:
        raise RuntimeError("Run this decoding method before inspecting its answers.")
    run_dir = RUN_DIRS[method]
    metrics = json.loads((run_dir / "metrics.json").read_text())
    records = [json.loads(line) for line in (run_dir / "predictions.jsonl").read_text().splitlines()
               if line.strip()]
    print("Saved metrics:")
    print(json.dumps(metrics, indent=2))
    if not records:
        print("No completed predictions are available.")
        return []
    selected = random.sample(records, min(count, len(records)))
    print(f"\nShowing saved predictions from: {run_dir}")
    for row in selected:
        question = row["prompt"].split("\n\nQuestion:\n", 1)[-1]
        print("=" * 80)
        print(f"Question ({row['example_id']}):\n{question}")
        print(f"\nModel response:\n{row['raw_output']}")
        print(f"\nExtracted answer: {row['predicted_answer']}")
        print(f"Correct answer:   {row['reference_answer']}")
        print(f"Correct:          {row['exact_match']}")
        if "votes" in row:
            print("Candidate answer votes:", row["votes"])
        print()
    return selected


## Decoding methods

Run each method independently. Every method uses the shared settings above and creates a separate experiment. Inference costs count every model forward, including all five candidates for Best-of-5.

### Confidence based decoding

Rank masked positions by the probabilities of their sampled tokens.

In [ ]:
run_decoding("confidence")


In [ ]:
show_random_answers("confidence")


### Random MDM decoding

Sample proposed tokens and choose positions to fill at random.

In [ ]:
run_decoding("random_mdm")


In [ ]:
show_random_answers("random_mdm")


### Top-K Margin decoding

Rank masked positions by the gap between their two highest token probabilities.

In [ ]:
run_decoding("top_k_margin")


In [ ]:
show_random_answers("top_k_margin")


### Best-of-5 decoding

Generate five confidence candidates with distinct seeds and select their most frequent normalized final answer.

In [ ]:
run_decoding("best_of_5")


In [ ]:
show_random_answers("best_of_5")


## Saved artifacts and larger evaluations

Runs are saved under `MyDrive/CSCI544-DLM/iLLaDA/GSM8K/<method>/<profile>/<settings>/<timestamp>/`.

- `metrics.json`: accuracy, parsing failures, actual model forwards, generation latency, throughput, GPU memory, and run status.
- `predictions.jsonl`: each question's prompt, raw response, normalized prediction and reference answer, correctness, seed, and model identity. Best-of-5 also saves all five responses and voting details.
- `config.json`, `manifest.json`, and `traces/`: resolved settings, pinned dataset examples, and token commitments.

For a larger evaluation, update `PROFILE` in the configuration cell to `"initial"` or `"full"`, keep `NUM_SAMPLES = None`, rerun that cell, and run the desired methods. All methods and both model families reuse the same frozen test examples. Accuracy requires an explicitly marked final `#### <number>` line; malformed answers count as incorrect. This notebook does not claim a paper reproduction score.

For CLI commands and algorithm details, see [the decoding README](https://github.com/CooL-Legend/CSCI544-DLM/blob/adi/src/dlm_project/decoding/README.md).